In [ ]:
import numpy as np
import skfuzzy as fuzz
import matplotlib.pyplot as plt

# Yaş için üyelik fonksiyonları
x_age = np.arange(0, 101, 1)
young = fuzz.trapmf(x_age, [0, 0, 30, 40])  # Genç
middle_aged = fuzz.trimf(x_age, [30, 50, 70])  # Orta yaşlı
old = fuzz.trapmf(x_age, [60, 70, 100, 100])  # Yaşlı

# PD-L1 skoru için üyelik fonksiyonları
x_pdl1 = np.arange(0, 101, 1)
low_pdl1 = fuzz.trapmf(x_pdl1, [0, 0, 20, 40])  # Düşük
medium_pdl1 = fuzz.trimf(x_pdl1, [30, 50, 70])  # Orta
high_pdl1 = fuzz.trapmf(x_pdl1, [60, 80, 100, 100])  # Yüksek

# Başarı oranı için üyelik fonksiyonları
x_success = np.arange(0, 101, 1)
low_success = fuzz.trapmf(x_success, [0, 0, 30, 40])  # Düşük başarı
medium_success = fuzz.trimf(x_success, [30, 50, 70])  # Orta başarı
high_success = fuzz.trapmf(x_success, [60, 80, 100, 100])  # Yüksek başarı

# Yaş ve PD-L1 skoru için fuzzy kuralları tanımlama
age_input = 20  # Örnek yaş
pdl1_input = 80 # Örnek PD-L1 skoru

# Üyelik fonksiyonlarıyla girdi verilerini bulanıklaştırma
age_level_young = fuzz.interp_membership(x_age, young, age_input)
age_level_middle_aged = fuzz.interp_membership(x_age, middle_aged, age_input)
age_level_old = fuzz.interp_membership(x_age, old, age_input)

pdl1_level_low = fuzz.interp_membership(x_pdl1, low_pdl1, pdl1_input)
pdl1_level_medium = fuzz.interp_membership(x_pdl1, medium_pdl1, pdl1_input)
pdl1_level_high = fuzz.interp_membership(x_pdl1, high_pdl1, pdl1_input)

# Kurallar: Fuzzy mantık kurallarını tanımlıyoruz
rule1 = np.fmax(age_level_young, pdl1_level_high)
rule2 = np.fmax(age_level_middle_aged, pdl1_level_medium)
rule3 = np.fmax(age_level_old, pdl1_level_low)

# Kurallara göre başarı oranı hesaplama
success_level_low = np.fmin(rule3, low_success)
success_level_medium = np.fmin(rule2, medium_success)
success_level_high = np.fmin(rule1, high_success)

# Fuzzy sonuçları birleştirme (defuzzification)
aggregated = np.fmax(success_level_low, np.fmax(success_level_medium, success_level_high))

# Sonuçları defuzzify ederek kesin başarı oranını elde etme
success = fuzz.defuzz(x_success, aggregated, 'centroid')

print(f"İmmünoterapinin başarı oranı: {success}%")

# Grafiksel gösterim
fig, (ax0, ax1, ax2) = plt.subplots(nrows=1, ncols=3, figsize=(10, 3))

ax0.plot(x_age, young, label="Genç")
ax0.plot(x_age, middle_aged, label="Orta Yaşlı")
ax0.plot(x_age, old, label="Yaşlı")
ax0.set_title('Yaş Üyelik Fonksiyonları')
ax0.legend()

ax1.plot(x_pdl1, low_pdl1, label="Düşük")
ax1.plot(x_pdl1, medium_pdl1, label="Orta")
ax1.plot(x_pdl1, high_pdl1, label="Yüksek")
ax1.set_title('PD-L1 Üyelik Fonksiyonları')
ax1.legend()

ax2.plot(x_success, low_success, label="Düşük")
ax2.plot(x_success, medium_success, label="Orta")
ax2.plot(x_success, high_success, label="Yüksek")
ax2.set_title('Başarı Oranı Üyelik Fonksiyonları')
ax2.legend()

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import skfuzzy as fuzz
import matplotlib.pyplot as plt

# Veriyi yükle
data = pd.read_csv("data/combined_data.csv")
print(data.head())
print(data.isnull().sum())

# Eksik verileri doldurma
data.fillna(data.select_dtypes(include=['number']).mean(), inplace=True)

# Sadece gerekli sütunları seç
data = data[['Age', 'Clinically reported PD-L1 score', 'Durable clinical response']]

# Normalleştirme başlatıcı
scaler = MinMaxScaler()

# Normalleştirme işlemi
data[['Age', 'Clinically reported PD-L1 score']] = scaler.fit_transform(data[['Age', 'Clinically reported PD-L1 score']])
print(data.head())

# Yaş için üyelik fonksiyonları (Genç, orta yaşlı, yaşlı)
x_age = np.arange(0, 1.1, 0.1)
young = fuzz.trimf(x_age, [0, 0, 0.3])  # Genç
middle_aged = fuzz.trimf(x_age, [0.3, 0.5, 0.7])  # Orta yaşlı
old = fuzz.trimf(x_age, [0.5, 0.7, 1])  # Yaşlı


# PD-L1 skoru için üyelik fonksiyonları (Düşük, Orta, Yüksek)
x_pdl1 = np.arange(0, 1.1, 0.1)  # Bu doğru aralık
low_pdl1 = fuzz.trimf(x_pdl1, [0, 0, 0.4])  # Düşük
medium_pdl1 = fuzz.trimf(x_pdl1, [0.3, 0.5, 0.7])  # Orta
high_pdl1 = fuzz.trimf(x_pdl1, [0.6, 0.8, 1])  # Yüksek

# Başarı oranı için üyelik fonksiyonları
x_success = np.arange(0, 1.1, 0.1)  # Başarı oranı aralığı
low_success = fuzz.trimf(x_success, [0, 0, 0.3])  # Düşük başarı
medium_success = fuzz.trimf(x_success, [0.3, 0.5, 0.7])  # Orta başarı
high_success = fuzz.trimf(x_success, [0.6, 0.8, 1])  # Yüksek başarı

# Yaş ve PD-L1 skoru için fuzzy üyelik fonksiyonlarıyla giriş verilerini bulanıklaştırma
age_input = 0.0  # Örneğin
pdl1_input = 0.5

# Üyelik derecelerini hesaplama
age_level_young = fuzz.interp_membership(x_age, young, age_input)
age_level_middle_aged = fuzz.interp_membership(x_age, middle_aged, age_input)
age_level_old = fuzz.interp_membership(x_age, old, age_input)

pdl1_level_low = fuzz.interp_membership(x_pdl1, low_pdl1, pdl1_input)
pdl1_level_medium = fuzz.interp_membership(x_pdl1, medium_pdl1, pdl1_input)
pdl1_level_high = fuzz.interp_membership(x_pdl1, high_pdl1, pdl1_input)

# Fuzzy kurallarına göre başarı oranlarını hesaplama
rule1 = np.fmin(age_level_young, pdl1_level_high)  # Genç ve yüksek PD-L1
rule2 = np.fmin(age_level_middle_aged, pdl1_level_medium)  # Orta yaş ve orta PD-L1
rule3 = np.fmin(age_level_old, pdl1_level_low)  # Yaşlı ve düşük PD-L1

# Ağırlıklı başarı oranları
weight_rule1 = 0.8  # Genç ve yüksek PD-L1: Yüksek ağırlık
weight_rule2 = 0.5 # Orta yaş ve orta PD-L1: Orta ağırlık
weight_rule3 = 0.4  # Yaşlı ve düşük PD-L1: Orta ağırlık

# Ağırlıklı başarı oranları
success_level_low = np.fmin(rule1, low_success) * weight_rule1
success_level_medium = np.fmin(rule2, medium_success) * weight_rule2
success_level_high = np.fmin(rule3, high_success) * weight_rule3

# Başarı oranlarını birleştirme (agregasyon)
aggregated = np.fmax(success_level_low, np.fmax(success_level_medium, success_level_high))

# Defuzzification (Kesin sonuç)
if np.sum(aggregated) == 0:
    success = 0  # Eğer hiç uygun değer yoksa başarı oranı sıfır olur.
else:
    success = fuzz.defuzz(x_success, aggregated, 'centroid')

print(f"İmmünoterapi başarı oranı: {success * 100:.2f}%")

# Grafiksel gösterim
fig, (ax0, ax1, ax2) = plt.subplots(nrows=1, ncols=3, figsize=(10, 3))

# Yaş üyelik fonksiyonlarını çizme
ax0.plot(x_age, young, label="Genç")
ax0.plot(x_age, middle_aged, label="Orta Yaşlı")
ax0.plot(x_age, old, label="Yaşlı")
ax0.set_title('Yaş Üyelik Fonksiyonları')
ax0.legend()

# PD-L1 üyelik fonksiyonlarını çizme
ax1.plot(x_pdl1, low_pdl1, label="Düşük")
ax1.plot(x_pdl1, medium_pdl1, label="Orta")
ax1.plot(x_pdl1, high_pdl1, label="Yüksek")
ax1.set_title('PD-L1 Üyelik Fonksiyonları')
ax1.legend()

# Başarı üyelik fonksiyonlarını çizme
ax2.plot(x_success, low_success, label="Düşük")
ax2.plot(x_success, medium_success, label="Orta")
ax2.plot(x_success, high_success, label="Yüksek")
ax2.set_title('Başarı Üyelik Fonksiyonları')
ax2.legend()

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

# Veri setini yükleyin (örnek)
df = pd.read_csv('data/combined_data.csv')  # veri setinizi burada yükleyin

# 'Age' sütunundaki en düşük ve en yüksek değeri bulun
min_pdl1 = df['Clinically reported PD-L1 score'].min()
max_pdl1 = df['Clinically reported PD-L1 score'].max()
min_age = df['Age'].min()
max_age = df['Age'].max()

print(f"En düşük yaş: {min_age}")
print(f"En yüksek yaş: {max_age}")
print(f"En düşük pdl1: {min_pdl1}")
print(f"En yüksek pdl1: {max_pdl1}")
